# Preprocessing notebook
Run notebooks in the following order:\
`preprocessing.ipynb -> imputation.ipynb -> dimensionality_reduction.ipynb -> gmm_clustering.ipynb`

In [ ]:
from __future__ import annotations

import itertools
import typing
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.io.stata import StataReader
from sklearn.model_selection import train_test_split

from utils.equivalent_cat_utils import EquivalenceRule, ImplicationRule, enforce_rules
from utils.preprocessing_utils import (
    convert_data_types,
    create_vars_info,
    remove_variables,
    sort_groups_vars_info,
    sort_types_vars_info,
)
from utils.saving_utils import save_data

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Read in data
data_path = Path("data/raw/ESA1821_neu_SO1.dta")

data_reader = pd.read_stata(data_path, iterator=True, convert_categoricals=False, convert_missing=False)
data_reader = typing.cast(StataReader, data_reader)
value_labels = data_reader.value_labels()
var_labels = data_reader.variable_labels()
with data_reader:
    data_full: pd.DataFrame = data_reader.read()  # type: ignore

# Set outputs folder
outputs_folder = "data/preprocessed"

# Define discovery and validation data split
index_discov, _ = train_test_split(
    data_full.index, test_size=0.5, random_state=np.random.RandomState(np.random.default_rng(3464).bit_generator)
)
# discov_mask = data_full.index.isin(index_discov)  # Random 50/50 split
discov_mask = data_full["zeit"] == 2018  # Discovery data is ESA 2018 and validation data is ESA 2021

In [ ]:
# Create information about variables
vars_info_full = create_vars_info(var_labels, value_labels)

misc_vars = ["wichtn", "sds_c", "sds_k", "sds_a"]

general_vars = ["fall", "zeit", "stadt_bl", "int_art"]

socioecon_vars = [
    "alter", "ges", "hh", "hh14", "allein", "staat", "Beruf", "erwerb", "fam", "schule", "hne", "westost_2",
]  # fmt: skip

alcohol_vars = [
    "wein30gr", "mish30gr", "bier30gr", "spir30gr", "alk30gr", "alk30kat", "alkkons", "alkkon", "binge", "bierkons",
    "spirkons", "mishkons", "hazar", "harmf", "alkrisk", "dsm_al", "dsm_al_m", "alk_kat", "vx210_2",
]  # fmt: skip

nicotine_vars = [
    "akt_rau", "typ", "Anzahl", "fag_sch", "ezp30", "typ_ezig", "dsm_ta", "psecdi_sch", "hnb30", "hnb_fag_sch",
    "typ_hnb", "typ_shish",
]  # fmt: skip

other_drugs_vars = [
    "hasch_12", "ecsta_12", "lsd_12", "heroi_12", "andop_12", "kokai_12", "crack_12", "pilz_12", "nps_12", "amp_12",
    "meth_12", "fr12_he", "fr12_ha1", "fr12_au1", "fr12_ec1", "fr12_ls1", "fr12_ao1", "fr12_ko1", "fr12_cr1",
    "fr12_sc1", "fr12_pi1", "fr12_nps1", "fr12_meth1", "dsm_ha", "dsm_au", "dsm_kc", "dsm_ha_m", "dsm_au_m", "dsm_kc_m",
    "Miss_Abh_ha", "Miss_Abh_kc", "Miss_Abh_am", "Miss_Abh_dro",
]  # fmt: skip

medication_vars = [
    "schm_30", "schl_30", "ber_30", "anr_30", "app_30", "depr_30", "neur_30", "anab_30", "m_schm", "m_schl", "m_ber",
    "m_anr", "m_app", "m_depr", "m_neur", "m_anab", "t_schm", "t_schl", "t_ber", "t_anr", "t_app", "t_depr", "t_neur",
    "t_anab", "anr_12", "app_12", "anab_12", "dsm_schm", "dsm_schm_m", "rist", "dsm_schl", "dsm_ber", "dsm_schl_m",
    "dsm_ber_m", "pomi", "rist_noa",
]  # fmt: skip

# Assign group names
groups_by_name = {
    "misc": misc_vars, "general": general_vars, "socioecon": socioecon_vars, "alcohol": alcohol_vars,
    "nicotine": nicotine_vars, "other_drugs": other_drugs_vars, "medication": medication_vars,
}  # fmt: skip

# Define variable types
all_na_vars = [var for var in data_full.columns if data_full[var].isna().all()]

continuous_vars = [
    "wichtn", "alter", "hh", "hh14", "wein30gr", "mish30gr", "bier30gr", "spir30gr", "alk30gr", "Anzahl", "pomi",
]  # fmt: skip

ordinal_vars = [
    "hne", "alk30kat", "alkkons", "alkkon", "binge", "alkrisk", "alk_kat", "akt_rau", "fag_sch", "psecdi_sch",
    "hnb_fag_sch", "fr12_he", "fr12_ha1", "fr12_au1", "fr12_ec1", "fr12_ls1", "fr12_ao1", "fr12_ko1", "fr12_cr1",
    "fr12_sc1", "fr12_pi1", "fr12_nps1", "fr12_meth1", "sds_c", "sds_k", "sds_a",
]  # fmt: skip

binary_vars = [
    var
    for var in data_full.columns
    if var not in itertools.chain(all_na_vars, ordinal_vars, continuous_vars)
    and data_full[var].nunique(dropna=True) == 2
    # and set(value_labels.get(var, {}).values()) in [set(["nein", "ja"]), set(["Nein", "Ja"])]
]

nominal_vars = [
    var
    for var in data_full.columns
    if var not in itertools.chain(all_na_vars, ordinal_vars, continuous_vars, binary_vars)
]

# Assign type names
types_by_name = {
    "all_na": all_na_vars, "continuous": continuous_vars, "binary": binary_vars, "nominal": nominal_vars,
    "ordinal": ordinal_vars,
}  # fmt: skip

# Add group and type info to information about variables
vars_info_full = sort_groups_vars_info(vars_info_full, groups_by_name)
vars_info_full = sort_types_vars_info(vars_info_full, types_by_name)

In [ ]:
# Create category codings
data_full = convert_data_types(data_full, vars_info_full)

# Correct coding errors
data_full = data_full.assign(
    **{
        # Remove invalid value of akt_rau that occurs only once
        "akt_rau": lambda x: x["akt_rau"].cat.remove_categories(
            x["akt_rau"].value_counts().loc[lambda x: x == 1].index
        ),
        # Reorder the levels of alkrisk
        "alkrisk": lambda x: x["alkrisk"].cat.reorder_categories([1, 2, 3, 6, 4, 5]),
    }
)

In [ ]:
# Impute using structural rules and drop invalid combinations

# implied by one month abstinence from alcohol
implied_alcohol = {
    "wein30gr": 0, "mish30gr": 0, "bier30gr": 0, "spir30gr": 0, "alk30gr": 0, "alk30kat": 0, "alkkons": [0, 1, 2],
    "alkkon": [0, 1, 2], "binge": 1, "bierkons": 0, "spirkons": 0, "mishkons": 0, "alkrisk": [1, 2, 3], "vx210_2": 0,
}  # fmt: skip

# In the following `implies` is always abstinence from a substance `is_implied` are conclusions that necessarily follow.
rules_alcohol = [
    # whole life
    ImplicationRule(
        implies={"alkkons": 0}, is_implied=implied_alcohol | {"alkkons": 0, "alkkon": 0, "alkrisk": 1, "alk_kat": 0}
    ),
    ImplicationRule(
        implies={"alkkon": 0}, is_implied=implied_alcohol | {"alkkons": 0, "alkkon": 0, "alkrisk": 1, "alk_kat": 0}
    ),
    ImplicationRule(
        implies={"alkrisk": 1}, is_implied=implied_alcohol | {"alkkons": 0, "alkkon": 0, "alkrisk": 1, "alk_kat": 0}
    ),

    # one year but not whole life
    ImplicationRule(
        implies={"alkkons": 1}, is_implied=implied_alcohol | {"alkkons": 1, "alkkon": 1, "alkrisk": 2, "alk_kat": 0}
    ),
    ImplicationRule(
        implies={"alkkon": 1}, is_implied=implied_alcohol | {"alkkons": 1, "alkkon": 1, "alkrisk": 2, "alk_kat": 0}
    ),
    ImplicationRule(
        implies={"alkrisk": 2}, is_implied=implied_alcohol | {"alkkons": 1, "alkkon": 1, "alkrisk": 2, "alk_kat": 0}
    ),

    # one year
    ImplicationRule(
        implies={"alkkons": [0, 1]},
        is_implied=implied_alcohol | {"alkkons": [0, 1], "alkkon": [0, 1], "alkrisk": [1, 2], "alk_kat": 0},
    ),
    ImplicationRule(
        implies={"alkkon": [0, 1]},
        is_implied=implied_alcohol | {"alkkons": [0, 1], "alkkon": [0, 1], "alkrisk": [1, 2], "alk_kat": 0},
    ),
    ImplicationRule(
        implies={"alkrisk": [1, 2]},
        is_implied=implied_alcohol | {"alkkons": [0, 1], "alkkon": [0, 1], "alkrisk": [1, 2], "alk_kat": 0},
    ),
    ImplicationRule(
        implies={"alk_kat": 0},
        is_implied=implied_alcohol | {"alkkons": [0, 1], "alkkon": [0, 1], "alkrisk": [1, 2], "alk_kat": 0},
    ),

    # one month but not one year
    ImplicationRule(implies={"alkkons": 2}, is_implied=implied_alcohol | {"alkkons": 2, "alkkon": 2, "alkrisk": 3}),
    ImplicationRule(implies={"alkkon": 2}, is_implied=implied_alcohol | {"alkkons": 2, "alkkon": 2, "alkrisk": 3}),
    ImplicationRule(implies={"alkrisk": 3}, is_implied=implied_alcohol | {"alkkons": 2, "alkkon": 2, "alkrisk": 3}),

    # one month
    ImplicationRule(implies={"alk30gr": 0}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alk30kat": 0}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alkkons": [0, 1, 2]}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alkkon": [0, 1, 2]}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alkrisk": [1, 2, 3]}, is_implied=implied_alcohol),
    ImplicationRule(implies={"alk_kat": 0}, is_implied=implied_alcohol),
    ImplicationRule(implies={"vx210_2": 0}, is_implied=implied_alcohol),
]  # fmt: skip

rules_nicotine = [
    EquivalenceRule({"ezp30": 0, "typ_ezig": 0}),
    EquivalenceRule({"ezp30": 1, "typ_ezig": 1}),
    EquivalenceRule({"hnb30": 0, "typ_hnb": 0}),
    EquivalenceRule({"hnb30": 1, "typ_hnb": 1}),
]

rules_medication = [
    # one year
    ImplicationRule(implies={"anr_12": 0}, is_implied={"m_anr": 0, "t_anr": 0}),
    ImplicationRule(implies={"app_12": 0}, is_implied={"m_app": 0, "t_app": 0}),
    ImplicationRule(implies={"anab_12": 0}, is_implied={"m_anab": 0, "t_anab": 0}),

    # one month
    ImplicationRule(implies={"schm_30": 0}, is_implied={"m_schm": 0, "t_schm": 0}),
    ImplicationRule(implies={"schl_30": 0}, is_implied={"m_schl": 0, "t_schl": 0}),
    ImplicationRule(implies={"ber_30": 0}, is_implied={"m_ber": 0, "t_ber": 0}),
    ImplicationRule(implies={"anr_30": 0}, is_implied={"m_anr": 0, "t_anr": 0}),
    ImplicationRule(implies={"app_30": 0}, is_implied={"m_app": 0, "t_app": 0}),
    ImplicationRule(implies={"depr_30": 0}, is_implied={"m_depr": 0, "t_depr": 0}),
    ImplicationRule(implies={"neur_30": 0}, is_implied={"m_neur": 0, "t_neur": 0}),
    ImplicationRule(implies={"anab_30": 0}, is_implied={"m_anab": 0, "t_anab": 0}),

    # one week
    ImplicationRule(implies={"m_schm": 0}, is_implied={"t_schm": 0}),
    ImplicationRule(implies={"m_schl": 0}, is_implied={"t_schl": 0}),
    ImplicationRule(implies={"m_ber": 0}, is_implied={"t_ber": 0}),
    ImplicationRule(implies={"m_anr": 0}, is_implied={"t_anr": 0}),
    ImplicationRule(implies={"m_app": 0}, is_implied={"t_app": 0}),
    ImplicationRule(implies={"m_depr": 0}, is_implied={"t_depr": 0}),
    ImplicationRule(implies={"m_neur": 0}, is_implied={"t_neur": 0}),
    ImplicationRule(implies={"m_anab": 0}, is_implied={"t_anab": 0}),
]  # fmt: skip

rules_other_drugs = [
    # one year
    EquivalenceRule({"hasch_12": 0, "fr12_ha1": 1}),
    EquivalenceRule({"ecsta_12": 0, "fr12_ec1": 1}),
    EquivalenceRule({"lsd_12": 0, "fr12_ls1": 1}),
    EquivalenceRule({"heroi_12": 0, "fr12_he": 1}),
    EquivalenceRule({"andop_12": 0, "fr12_ao1": 1}),
    EquivalenceRule({"kokai_12": 0, "fr12_ko1": 1}),
    EquivalenceRule({"crack_12": 0, "fr12_cr1": 1}),
    EquivalenceRule({"pilz_12": 0, "fr12_pi1": 1}),
    EquivalenceRule({"nps_12": 0, "fr12_nps1": 1}),
    EquivalenceRule({"amp_12": 0, "fr12_au1": 1}),
    EquivalenceRule({"meth_12": 0, "fr12_meth1": 1}),

    # additional
    ImplicationRule(implies={"dsm_ha": 0, "dsm_ha_m": 0}, is_implied={"Miss_Abh_ha": 0}),
    ImplicationRule(implies={"dsm_ha": 1, "dsm_ha_m": 0}, is_implied={"Miss_Abh_ha": 1}),
    ImplicationRule(implies={"dsm_ha": 0, "dsm_ha_m": 1}, is_implied={"Miss_Abh_ha": 1}),
    ImplicationRule(implies={"dsm_ha": 1, "dsm_ha_m": 1}, is_implied={"Miss_Abh_ha": 1}),
]  # fmt: skip

rules = [*rules_alcohol, *rules_nicotine, *rules_medication, *rules_other_drugs]
data_imputed = enforce_rules(data_full, rules=rules, na_strategy="impute", failure_strategy="drop", verbose=True)

In [ ]:
# Recode categories
data_full = data_full.assign(
    **{
        # Retain the 10 most common cities and lump the others into one category
        # (more than 10 are retained if there are ties)
        "stadt_bl": lambda x: (
            x["stadt_bl"]
            .map(lambda val: val if val in x["stadt_bl"].value_counts().nlargest(n=10, keep="all").index else "other")
            .astype("category")
        ),
    }
)

In [ ]:
# Remove variables

# Variables that are NA everywhere in the discovery or validation dataset
na_vars: list[str] = (
    (data_full[discov_mask].isnull().all() | data_full[~discov_mask].isnull().all()).loc[lambda x: x].index.to_list()
)
print(
    f"Dropped the following variables from the dataset because they are missing everywhere in the discovery or "
    f"validation dataset: {na_vars}"
)
excluded_vars = [
    # alcohol
    "alk30gr", "alk30kat", "alkkon", "alkrisk",  # "alk_kat"
    "vx210_2",

    # nicotine
    "typ_ezig", "typ_hnb", "fag_sch", "hnb_fag_sch", "psecdi_sch",  # "Anzahl",

    # other_drugs
    "hasch_12", "ecsta_12", "lsd_12", # "heroi_12",
    "andop_12", "kokai_12", "crack_12", "pilz_12", # "nps_12",
    "amp_12", "meth_12", "fr12_he",

    # medication
    "m_schm", "m_schl", "m_ber", "m_anr", "m_app", "m_depr", "m_neur", "m_anab", "t_schm", "t_schl", "t_ber", "t_anr",
    "t_app", "t_depr", "t_neur", "t_anab",  # "pomi",
]  # fmt: skip
drop_vars = [*misc_vars, *general_vars, *socioecon_vars, *excluded_vars, *na_vars]
data_reduced, vars_info_reduced = remove_variables(data_full, vars_info_full, drop_vars=drop_vars)

In [ ]:
# Discovery/validation split
data_discov_full = data_full[discov_mask]
data_valid_full = data_full[~discov_mask]

data_discov_reduced = data_reduced[discov_mask]
data_valid_reduced = data_reduced[~discov_mask]

In [ ]:
# Save full data and information about variables
save_data(data_full, Path(outputs_folder, "data_full.pkl"))
save_data(vars_info_full, Path(outputs_folder, "vars_info_full.pkl"))

# Save reduced data and reduced information about variables
save_data(data_reduced, Path(outputs_folder, "data_reduced.pkl"))
save_data(vars_info_reduced, Path(outputs_folder, "vars_info_reduced.pkl"))

# Save full and reduced discovery and validation data
save_data(data_discov_full, Path(outputs_folder, "data_discov_full.pkl"))
save_data(data_valid_full, Path(outputs_folder, "data_valid_full.pkl"))

save_data(data_discov_reduced, Path(outputs_folder, "data_discov_reduced.pkl"))
save_data(data_valid_reduced, Path(outputs_folder, "data_valid_reduced.pkl"))